# Comprehensive Technical Notes on Multiprocessing in Python

## Table of Contents

1. [Introduction](#introduction)
2. [The Process Class](#the-process-class)
3. [Daemon Processes](#daemon-processes)
4. [Passing Arguments and Subclassing Process](#passing-arguments-and-subclassing-process)
5. [Starting Methods: fork, spawn, forkserver](#starting-methods-fork-spawn-forkserver)
6. [Synchronization Primitives](#synchronization-primitives)
   - [Lock](#lock)
   - [RLock](#rlock)
   - [Semaphore](#semaphore)
   - [Event](#event)
   - [Condition](#condition)
   - [Barrier](#barrier)
7. [Shared State: Value and Array](#shared-state-value-and-array)
8. [Manager (Server Process)](#manager-server-process)
9. [Communication Between Processes: Queues and Pipes](#communication-between-processes-queues-and-pipes)
10. [Process Pools (multiprocessing.Pool)](#process-pools-multiprocessingpool)
11. [ProcessPoolExecutor (concurrent.futures)](#processpoolexecutor-concurrentfutures)
12. [Handling Exceptions and Process Termination](#handling-exceptions-and-process-termination)
13. [Debugging and Profiling](#debugging-and-profiling)
14. [Best Practices and Common Pitfalls](#best-practices-and-common-pitfalls)
15. [Comparison: multiprocessing vs threading vs asyncio](#comparison-multiprocessing-vs-threading-vs-asyncio)
16. [Conclusion](#conclusion)

---

## Introduction

The `multiprocessing` module allows Python programs to create and manage **separate operating system processes**, bypassing the Global Interpreter Lock (GIL) and achieving true parallel execution on multi‑core CPUs. Each process has its own Python interpreter and memory space; data is exchanged via explicit serialisation (pickle) or shared‑memory constructs.

**Use multiprocessing for CPU‑bound tasks** (e.g., heavy calculations, image/video processing, data crunching).  
For I/O‑bound concurrency, consider `threading` or `asyncio` – processes introduce higher overhead.

```python
import multiprocessing
import time

def cpu_task(n):
    """A CPU‑intensive function."""
    total = 0
    for i in range(n):
        total += i * i
    return total

if __name__ == '__main__':
    t0 = time.time()
    # Run sequentially
    cpu_task(50_000_000)
    cpu_task(50_000_000)
    print(f"Sequential: {time.time() - t0:.2f}s")

    t0 = time.time()
    # Run in two processes
    p1 = multiprocessing.Process(target=cpu_task, args=(50_000_000,))
    p2 = multiprocessing.Process(target=cpu_task, args=(50_000_000,))
    p1.start(); p2.start()
    p1.join(); p2.join()
    print(f"Parallel:   {time.time() - t0:.2f}s")
```

Typical output: sequential ~1.5s, parallel ~0.8s – real speedup.

---

## The Process Class

```python
import multiprocessing as mp
import os

def worker(name):
    print(f"Worker {name} running in process ID {os.getpid()}, parent {os.getppid()}")

if __name__ == '__main__':
    p = mp.Process(target=worker, args=("A",))
    p.start()
    p.join()
    print("Main process finished")
```

- `p.start()` launches a new child process; the target function is executed in a fresh interpreter.
- `p.join(timeout=None)` blocks the calling process until the child finishes.
- `p.is_alive()` checks if the process is still running.
- `p.terminate()` sends a `SIGTERM` (Unix) / `TerminateProcess` (Windows) – not a clean shutdown.
- `p.kill()` sends `SIGKILL` – immediate, no cleanup.
- `p.exitcode` returns the exit status (`None` if still running, `0` on success, positive/negative on error).

---

## Daemon Processes

A **daemon process** is terminated abruptly when the main program exits (all non‑daemon child processes have finished). Useful for background tasks that do not require graceful cleanup.

```python
import multiprocessing as mp
import time

def background_task():
    while True:
        print("Daemon alive")
        time.sleep(1)

if __name__ == '__main__':
    d = mp.Process(target=background_task, daemon=True)
    d.start()
    time.sleep(3)
    print("Main exits – daemon will be killed")
```

**Warning**: Daemon processes cannot have children. They may leave resources in an undefined state.

---

## Passing Arguments and Subclassing Process

### Subclassing Process

```python
class ComputeProcess(mp.Process):
    def __init__(self, number):
        super().__init__()
        self.number = number

    def run(self):
        result = self.number ** 2
        print(f"{self.number} squared is {result}")

if __name__ == '__main__':
    p = ComputeProcess(9)
    p.start()
    p.join()
```

Override `run()`; the constructor can accept additional arguments (must call `super().__init__()`).

### Passing mutable data

Arguments must be picklable. Mutable objects are copied between processes by pickling, so changes in one process are **not** visible in another unless using shared state (see later).

```python
def append_to_list(lst, value):
    lst.append(value)
    print(f"Inside process: {lst}")

if __name__ == '__main__':
    shared = [1, 2]
    p = mp.Process(target=append_to_list, args=(shared, 3))
    p.start()
    p.join()
    print(f"Main process: {shared}")  # still [1, 2]
```

---

## Starting Methods: fork, spawn, forkserver

Python 3.8+ supports three start methods that define how child processes are created:

| Method | Behaviour | Default on |
|--------|-----------|-------------|
| `fork` | Copies the parent’s entire memory space (copy‑on‑write). Fast, but can be unsafe (locks/threads duplicated). | Unix |
| `spawn` | Starts a fresh Python interpreter, imports the main module, then calls the target. Slower but most reliable. | Windows/macOS |
| `forkserver` | Uses a server process as a clean template for new processes (available on Unix). | |

**Set the start method once at the beginning of your program:**

```python
import multiprocessing as mp

mp.set_start_method('spawn', force=True)
# or use get_context('spawn').Process() / .Pool() to create context-specific objects
```

When using `spawn` or `forkserver`, **always protect the entry point** with:

```python
if __name__ == '__main__':
    # create processes here
```

This prevents infinite subprocess spawning.

---

## Synchronization Primitives

`multiprocessing` provides analogues of `threading` primitives, but they work across processes via semaphores, shared memory, or other OS mechanisms.

### Lock

Mutual exclusion across processes.

```python
import multiprocessing as mp

def increment(counter, lock):
    for _ in range(100000):
        with lock:
            counter.value += 1

if __name__ == '__main__':
    counter = mp.Value('i', 0)        # shared integer
    lock = mp.Lock()
    procs = [mp.Process(target=increment, args=(counter, lock)) for _ in range(2)]
    for p in procs: p.start()
    for p in procs: p.join()
    print(counter.value)  # 200000
```

**Lock context manager** (`with lock`) is available.

### RLock (Re‑entrant Lock)

Allows the **same process** to acquire it multiple times without blocking itself.

```python
rlock = mp.RLock()
with rlock:
    with rlock:
        print("Re-entrant acquired")
```

### Semaphore

Limits access to a resource, e.g., maximum concurrent processes.

```python
sem = mp.Semaphore(3)   # max 3 processes at a time
with sem:
    # critical section
    pass
```

`BoundedSemaphore` also catches unmatched releases.

### Event

A flag that can be set, cleared, and waited on.

```python
def waiter(event):
    print("Waiting for event...")
    event.wait()
    print("Event received")

def setter(event):
    time.sleep(2)
    event.set()

if __name__ == '__main__':
    evt = mp.Event()
    mp.Process(target=waiter, args=(evt,)).start()
    mp.Process(target=setter, args=(evt,)).start()
```

### Condition

A condition variable combining a lock and wait/notify.

```python
def producer(cond, queue):
    with cond:
        queue.put('item')
        cond.notify()

def consumer(cond, queue):
    with cond:
        while queue.empty():
            cond.wait()
        item = queue.get()
```

(Here `queue` would be a `multiprocessing.Queue` or `Manager.Queue`.)

### Barrier

All processes must call `wait()` before any can proceed.

```python
barrier = mp.Barrier(3)
def task():
    print("Before barrier")
    barrier.wait()
    print("After barrier")
```

---

## Shared State: Value and Array

`multiprocessing.Value` and `Array` create shared c‑types objects stored in a shared memory block.

```python
import multiprocessing as mp

# Value: typecode 'i' (int), 'd' (double), 'c' (char)
val = mp.Value('i', 0)

# Array: typecode, initial value or length
arr = mp.Array('d', [1.0, 2.5, 3.7])
arr2 = mp.Array('i', 5)   # length 5, all zeros

def modify(val, arr):
    val.value = 42
    arr[0] = 99.9

if __name__ == '__main__':
    p = mp.Process(target=modify, args=(val, arr))
    p.start()
    p.join()
    print(val.value)      # 42
    print(arr[:])         # [99.9, 2.5, 3.7]
```

**Note**: Only basic c‑types are supported. Locking is **not** automatic; for atomic operations use `multiprocessing.Value('i', lock=True)` (default). Array access is **not** atomically locked for compound operations – use a separate lock when necessary.

---

## Manager (Server Process)

A `Manager` runs a separate server process that holds complex objects (lists, dicts, queues, etc.). Other processes access them via proxies.

```python
import multiprocessing as mp

def worker(d, lst, q):
    d['key'] = 'value'
    lst.append(10)
    q.put("Hello")

if __name__ == '__main__':
    with mp.Manager() as manager:
        d = manager.dict()
        lst = manager.list()
        q = manager.Queue()     # Manager’s queue, not mp.Queue

        p = mp.Process(target=worker, args=(d, lst, q))
        p.start()
        p.join()

        print(d['key'])         # value
        print(lst)              # [10]
        print(q.get())          # Hello
```

The manager creates proxy objects that communicate via pickling. This approach is flexible but slower than raw `Value`/`Array` or `mp.Queue`.

**Common manager types**: `list`, `dict`, `Namespace`, `Queue`, `Event`, `Lock`, `Semaphore`, etc.

---

## Communication Between Processes: Queues and Pipes

### multiprocessing.Queue

A thread‑ and process‑safe FIFO queue based on pipes and locks.

```python
import multiprocessing as mp

def producer(q):
    for i in range(5):
        q.put(f"item-{i}")

def consumer(q):
    while True:
        item = q.get()
        if item is None:        # poison pill
            break
        print(f"Consumed {item}")

if __name__ == '__main__':
    q = mp.Queue()
    p1 = mp.Process(target=producer, args=(q,))
    p2 = mp.Process(target=consumer, args=(q,))
    p1.start(); p2.start()
    p1.join()
    q.put(None)      # send poison pill to consumer
    p2.join()
```

`Queue.get()` blocks until an item is available; use `get_nowait()` or `timeout`.

### Pipe

A simplex or duplex communication channel, faster than queue for two endpoints.

```python
parent_conn, child_conn = mp.Pipe()   # duplex by default

def sender(conn):
    conn.send("Hello")
    conn.send({"a": 1})
    conn.close()

def receiver(conn):
    print(conn.recv())  # Hello
    print(conn.recv())  # {'a': 1}
```

Pipes are **not** process‑safe when multiple processes read/write simultaneously – use a queue for fan‑out.

---

## Process Pools (multiprocessing.Pool)

`Pool` provides a convenient way to parallelize the execution of a function across multiple inputs, distributing work among a fixed number of worker processes.

```python
import multiprocessing as mp
import time

def square(x):
    time.sleep(0.1)   # simulate CPU work
    return x * x

if __name__ == '__main__':
    with mp.Pool(processes=4) as pool:
        # map – blocks until all results are ready, returns list
        results = pool.map(square, range(10))
        print(results)  # [0,1,4,9,...]

        # imap_unordered – iterator yielding results as they complete
        for res in pool.imap_unordered(square, range(10, 20)):
            print(res)

        # apply_async – submit a single task, get AsyncResult
        res = pool.apply_async(square, (100,))
        print(res.get(timeout=1))
```

**Variants**:

- `map_async`, `imap`, `starmap` (for functions taking multiple arguments: `f(a,b)`)
- `maxtasksperchild` – restart a worker after it completes a certain number of tasks, preventing memory leaks.

```python
pool = mp.Pool(processes=4, maxtasksperchild=100)
```

**Asynchronicity**: All `apply_async` / `map_async` return an `AsyncResult` object with methods:
- `get(timeout)`
- `ready()`
- `successful()`
- `wait(timeout)`

---

## ProcessPoolExecutor (concurrent.futures)

`concurrent.futures.ProcessPoolExecutor` provides a **higher‑level** interface, very similar to `ThreadPoolExecutor`.

```python
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

def cpu_intensive_task(n):
    return sum(i * i for i in range(n))

if __name__ == '__main__':
    with ProcessPoolExecutor(max_workers=4) as executor:
        # map returns results in input order
        results = executor.map(cpu_intensive_task, [10_000_000]*8)
        for i, val in enumerate(results):
            print(f"Task {i}: {val}")

        # submit and as_completed
        futures = {executor.submit(cpu_intensive_task, n): n for n in [5_000_000]*6}
        for future in as_completed(futures):
            n = futures[future]
            print(f"Task with {n} finished: {future.result()}")
```

**Advantages over multiprocessing.Pool**:

- Standard future interface (same as threads).
- Better exception propagation (exception raised in `future.result()`).
- Cleaner context manager.

**Note**: With `spawn` start method, the main module must be importable without side effects; wrap code in `if __name__ == '__main__':` and avoid global state that might be executed at import.

---

## Handling Exceptions and Process Termination

### Exceptions in child processes

If a process raises an unhandled exception, `join()` will not re‑raise it, but `p.exitcode` will be 1. With `Pool` or `ProcessPoolExecutor`, exceptions are propagated when calling `get()` on the result.

**Example**:

```python
def faulty(x):
    if x == 5:
        raise ValueError("Invalid input")
    return x * x

with mp.Pool(4) as pool:
    asyncs = [pool.apply_async(faulty, (i,)) for i in range(10)]
    for a in asyncs:
        try:
            print(a.get())
        except Exception as e:
            print(f"Caught: {e}")
```

### Terminating processes

`p.terminate()` abruptly stops the child. Use only as a last resort. Graceful shutdown should use a sentinel or an `Event`.

**Graceful stop using `Event`**:

```python
def worker(run_event):
    while run_event.is_set():
        # do work
        pass

if __name__ == '__main__':
    run_event = mp.Event()
    run_event.set()
    p = mp.Process(target=worker, args=(run_event,))
    p.start()
    time.sleep(2)
    run_event.clear()
    p.join()
```

---

## Debugging and Profiling

- **Logging**: Use `multiprocessing.get_logger()`, which produces logs if `log_to_stderr(logging.DEBUG)` is called.
- **Process names**: `mp.current_process().name` helps distinguish processes.
- **`traceback` in child**: Wrap your target function to log full tracebacks.
- **Profiling**: Tools like `py-spy` can profile subprocesses. For memory, `memory_profiler` works across processes.
- **Pdb**: Not trivial across processes; consider using `mp.Process(..., daemon=True)` and attaching a debugger to the child pid.

```python
import logging, multiprocessing as mp

logger = mp.log_to_stderr(logging.DEBUG)
```

---

## Best Practices and Common Pitfalls

1. **Protect the entry point**: Always use `if __name__ == '__main__':` guard for code that creates processes, especially on Windows/spawn.
2. **Avoid shared global state**: Each process has its own memory; global variables are not shared.
3. **Use process‑safe structures** for communication: `Queue`, `Pipe`, `Manager` proxies.
4. **Be mindful of pickling**: Functions, classes, and arguments must be picklable. Avoid using lambdas as pool callbacks (they are not picklable unless using 3rd‑party libraries like `dill`).
5. **Limit the number of processes** to the number of CPU cores (or slightly more for mixed I/O). `mp.cpu_count()` is your guide.
6. **Join all processes** to avoid zombie processes.
7. **Use `Pool` / `ProcessPoolExecutor`** instead of manually creating many processes to manage lifecycle.
8. **Beware of resource leaks**: If using `fork`, locks, threads, or file descriptors can be inherited in inconsistent states. `spawn` is safer.
9. **`None`‑sentinel** pattern for stopping consumer processes from a queue.
10. **Do not share a `Pipe` with more than two processes** – it can cause data corruption.

---

## Comparison: multiprocessing vs threading vs asyncio

| Feature               | multiprocessing          | threading                | asyncio                   |
|-----------------------|--------------------------|--------------------------|---------------------------|
| Parallelism           | True (separate processes)| Limited by GIL            | Cooperative concurrency   |
| Use case              | CPU‑bound                | I/O‑bound                | Highly concurrent I/O     |
| Memory overhead       | High (independent interp.)| Low (shared memory)     | Very low                  |
| Communication         | Queues, Pipes, Shared mem| Shared objects (locks)   | No shared state, queues   |
| Start method issues   | fork/spawn complexity    | None                     | None (single thread)      |
| Pickling required     | Yes (arguments, results) | No (direct object sharing)| No (tasks in same thread) |
| Learning curve        | Moderate                 | Low                      | Steep                     |

**Guideline**: CPU‑bound → `multiprocessing`; I/O‑bound with many tasks → `asyncio`; I/O with simple parallelism → `threading`.

---

## Conclusion

Python’s `multiprocessing` module enables genuine parallel execution of CPU‑intensive work by launching separate operating system processes. Through the `Process`, `Pool`, and `ProcessPoolExecutor` abstractions, you can distribute tasks efficiently across multiple cores. Synchronization primitives, shared memory (`Value`, `Array`), and managers provide robust inter‑process coordination. Despite the overhead of separate memory and serialisation, careful application yields dramatic speedups on multi‑core machines.

Always choose the appropriate parallel model for your workload: threads for I/O‑bound concurrency, asyncio for high‑concurrency networking, and multiprocessing for CPU‑bound parallelism. By adhering to best practices—especially around start methods and pickling—you can write clean, efficient, and production‑ready parallel Python programs.